# 06 Buyback Sentiment and Clarity

This notebook runs the buyback-focused research pipeline:

1. Load cleaned transcripts
2. Identify buyback mentions
3. Split prepared remarks and Q&A
4. Score FinBERT sentiment by section
5. Compute buyback-specific clarity features
6. Merge revenue surprise and returns
7. Bucket sentiment, clarity, and revenue surprise
8. Run event-study sorts and a baseline cross-sectional regression

Assumptions:
- `data/FINAL.csv` is the canonical cleaned input file
- Older data-pulling and cleaning scripts are WIP and should not be relied upon
- Component-level transcript fields are not currently bundled in `FINAL.csv`, so Q&A clarity remains WIP
- Model weights will be downloaded locally on first use if not already cached

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.binning import (
    create_sentiment_clarity_matrix,
    create_three_way_sort,
    format_results_table,
)
from src.config.settings import ProjectPaths
from src.data.buyback_events import build_buyback_pattern, extract_buyback_sentences, identify_buyback_transcripts
from src.data.final_dataset import get_final_column_map, summarize_final_dataset_capabilities
from src.data.load_transcript_components import (
    component_data_supports_qa_split,
    load_transcript_components,
    merge_transcript_components,
    resolve_transcript_component_path,
)
from src.data.load_transcripts import load_raw_transcripts
from src.data.qa_split import (
    flag_suspicious_qa_pairs,
    pair_questions_responses,
    split_prepared_qa,
    summarize_qa_pair_quality,
)
from src.data.revenue_surprise import (
    bucket_revenue,
    compute_ibes_revenue_surprise,
    compute_trend_revenue_surprise,
    merge_revenue_surprise,
)
from src.features.clarity import (
    bucket_clarity,
    compute_clarity_composite,
    compute_hedge_density,
    compute_modified_fog,
    compute_qa_relevance,
    compute_specificity,
)
from src.features.finbert_sentiment import bucket_sentiment, load_finbert_pipeline, score_transcript_sections
from src.finance.event_study import (
    IMAGE_SPEC_ESTIMATION_WINDOW,
    IMAGE_SPEC_ROBUSTNESS_WINDOWS,
    compute_caar_by_bins,
    run_event_study_from_wide_returns,
)

SAMPLE_MODE = True
SAMPLE_TRANSCRIPT_ROWS = 5000
SAMPLE_BUYBACK_CALLS = 250
MODEL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

paths = ProjectPaths()
paths

In [ ]:
transcripts = load_raw_transcripts(nrows=SAMPLE_TRANSCRIPT_ROWS if SAMPLE_MODE else None)

transcripts["call_date"] = pd.to_datetime(transcripts.get("call_date"), errors="coerce")
print("Canonical input file: data/FINAL.csv")
print(f"Sample mode: {SAMPLE_MODE} | transcript rows loaded: {len(transcripts):,}")
display(get_final_column_map(transcripts).head(20))
display(summarize_final_dataset_capabilities(transcripts))
transcripts.head()

In [ ]:
buyback_sample = identify_buyback_transcripts(transcripts)
buyback_sample = buyback_sample.copy()
if SAMPLE_MODE:
    buyback_sample = buyback_sample.head(SAMPLE_BUYBACK_CALLS).copy()
buyback_sample["buyback_sentences"] = buyback_sample["full_transcript_text"].map(extract_buyback_sentences)
buyback_sample["calendar_quarter"] = buyback_sample["call_date"].dt.to_period("Q")
print(f"Buyback sample rows: {len(buyback_sample):,}")

buyback_sample[["transcriptid", "companyname", "call_date", "buyback_sentences"]].head()

In [ ]:
component_path = None
buyback_transcript_ids = buyback_sample["transcriptid"].dropna().unique().tolist()
try:
    component_path = resolve_transcript_component_path()
    component_rows = load_transcript_components(
        component_path,
        transcript_ids=buyback_transcript_ids,
    )
except FileNotFoundError:
    component_rows = pd.DataFrame()

if not component_rows.empty and component_data_supports_qa_split(component_rows):
    component_source_label = component_rows.get("component_source", pd.Series(dtype=object)).dropna().astype(str)
    component_source_label = component_source_label.iloc[0] if not component_source_label.empty else "unknown"
    print(f"Using default Q&A source: {component_path} ({component_source_label})")
    print(
        f"Loaded {len(component_rows):,} component rows for {component_rows['transcriptid'].nunique():,} buyback transcripts."
    )
    prepared_df, qa_df = split_prepared_qa(component_rows)
    qa_pairs = pair_questions_responses(qa_df)
    qa_pair_quality = summarize_qa_pair_quality(qa_pairs)
    print(
        f"Prepared rows: {len(prepared_df):,} | Q&A rows: {len(qa_df):,} | "
        f"Paired analyst/executive exchanges: {len(qa_pairs):,}"
    )
    print(
        "Pair-quality check: "
        f"{qa_pair_quality['suspicious_pairs']:,} suspicious pairs "
        f"({qa_pair_quality['suspicious_share']:.1%} of all pairs)."
    )
else:
    print(
        "No usable component-level transcript file was found. "
        "Q&A speaker split and buyback clarity stay WIP until `data/interim/wrds_transcript_components.csv` is available."
    )
    prepared_df = pd.DataFrame(columns=buyback_sample.columns)
    qa_df = pd.DataFrame(columns=buyback_sample.columns)
    qa_pairs = pd.DataFrame(columns=["transcriptid", "question_text", "response_text"])
    qa_pair_quality = {
        "pair_count": 0,
        "suspicious_pairs": 0,
        "suspicious_share": 0.0,
        "median_question_words": 0.0,
        "median_response_words": 0.0,
    }

qa_pairs.head()


In [ ]:
print(f"FinBERT device: {MODEL_DEVICE}")
finbert_pipeline = load_finbert_pipeline(device=MODEL_DEVICE, batch_size=64)

prepared_text_by_transcript = (
    prepared_df.groupby("transcriptid")["componenttext"].apply(lambda values: " ".join(values.dropna().astype(str)))
    if not prepared_df.empty and "componenttext" in prepared_df.columns
    else pd.Series(dtype=object)
)
qa_text_by_transcript = (
    qa_df.groupby("transcriptid")["componenttext"].apply(lambda values: " ".join(values.dropna().astype(str)))
    if not qa_df.empty and "componenttext" in qa_df.columns
    else pd.Series(dtype=object)
)

sentiment_rows = []
for row in buyback_sample.itertuples(index=False):
    transcript_id = getattr(row, "transcriptid")
    sentiment_rows.append(
        {
            "transcriptid": transcript_id,
            **score_transcript_sections(
                transcript=getattr(row, "full_transcript_text", ""),
                prep_text=prepared_text_by_transcript.get(transcript_id, ""),
                qa_text=qa_text_by_transcript.get(transcript_id, ""),
                buyback_sentences=getattr(row, "buyback_sentences", []),
                pipeline_obj=finbert_pipeline,
            ),
        }
    )

sentiment_df = pd.DataFrame(sentiment_rows)
analysis_df = buyback_sample.merge(sentiment_df, on="transcriptid", how="left")
analysis_df[["transcriptid", "buyback_sentiment_mean", "buyback_sentiment_gap"]].head()

In [ ]:
buyback_pattern = build_buyback_pattern()

if not qa_pairs.empty:
    qa_pairs_flagged = flag_suspicious_qa_pairs(qa_pairs)
    qa_pairs_flagged["buyback_pair"] = (
        qa_pairs_flagged["question_text"].str.contains(buyback_pattern, na=False)
        | qa_pairs_flagged["response_text"].str.contains(buyback_pattern, na=False)
    )
    qa_buyback_pairs = qa_pairs_flagged.loc[qa_pairs_flagged["buyback_pair"]].copy()
    qa_pairs_clean = qa_pairs_flagged.loc[~qa_pairs_flagged["is_suspicious"]].copy()
    qa_buyback_pairs_clean = qa_buyback_pairs.loc[~qa_buyback_pairs["is_suspicious"]].copy()

    if len(qa_pairs_flagged):
        print(
            f"All Q&A pairs: {len(qa_pairs_flagged):,} | clean after filtering: {len(qa_pairs_clean):,} "
            f"({len(qa_pairs_clean) / len(qa_pairs_flagged):.1%} retained)"
        )
    else:
        print("All Q&A pairs: 0")

    if len(qa_buyback_pairs):
        print(
            f"Buyback Q&A pairs: {len(qa_buyback_pairs):,} | clean after filtering: {len(qa_buyback_pairs_clean):,} "
            f"({len(qa_buyback_pairs_clean) / len(qa_buyback_pairs):.1%} retained)"
        )
    else:
        print("Buyback Q&A pairs: 0")

    qa_buyback_pairs_clean["specificity"] = qa_buyback_pairs_clean["response_text"].map(compute_specificity)
    qa_buyback_pairs_clean["hedge_density"] = qa_buyback_pairs_clean["response_text"].map(compute_hedge_density)
    qa_buyback_pairs_clean["modified_fog"] = qa_buyback_pairs_clean["response_text"].map(compute_modified_fog)
    qa_buyback_pairs_clean["qa_relevance"] = [
        compute_qa_relevance(question, response)
        for question, response in zip(qa_buyback_pairs_clean["question_text"], qa_buyback_pairs_clean["response_text"])
    ]

    clarity_components = (
        qa_buyback_pairs_clean.groupby("transcriptid")[["specificity", "hedge_density", "modified_fog", "qa_relevance"]]
        .mean()
        .reset_index()
    )
    clarity_components["clarity_composite"] = compute_clarity_composite(
        clarity_components["specificity"],
        clarity_components["hedge_density"],
        clarity_components["modified_fog"],
        clarity_components["qa_relevance"],
    )
else:
    print("Clarity composite is WIP for FINAL.csv because buyback Q&A pairs are unavailable.")
    clarity_components = pd.DataFrame(
        columns=[
            "transcriptid",
            "specificity",
            "hedge_density",
            "modified_fog",
            "qa_relevance",
            "clarity_composite",
        ]
    )

analysis_df = analysis_df.merge(clarity_components, on="transcriptid", how="left")
analysis_df[["transcriptid", "clarity_composite"]].head()

In [ ]:
ibes_path = paths.interim_dir / "ibes_revenue_estimates.csv"

if ibes_path.exists():
    ibes_df = compute_ibes_revenue_surprise(pd.read_csv(ibes_path))
else:
    print("I/B/E/S revenue estimates are not bundled in FINAL.csv; using trend-based revenue surprise fallback.")
    ibes_df = pd.DataFrame(columns=["transcriptid", "ibes_revenue_surprise"])

trend_df = compute_trend_revenue_surprise(analysis_df)

analysis_df = merge_revenue_surprise(analysis_df, ibes_df, trend_df)
analysis_df["sentiment_bucket"] = bucket_sentiment(
    analysis_df["buyback_sentiment_mean"],
    groupby=analysis_df["calendar_quarter"],
)
analysis_df["clarity_bucket"] = bucket_clarity(
    analysis_df["clarity_composite"],
    groupby=analysis_df["calendar_quarter"],
)
analysis_df["revenue_bucket"] = bucket_revenue(analysis_df["revenue_surprise"])

analysis_df[["sentiment_bucket", "clarity_bucket", "revenue_bucket"]].head()

In [ ]:
print("Using the image-spec event-study design: estimation window [-15, -3] and post-event windows [+1, +3] and [+1, +5].")
print("This is supported directly from FINAL.csv via the wide ret_t... columns.")

event_results = run_event_study_from_wide_returns(
    analysis_df,
    estimation_window=IMAGE_SPEC_ESTIMATION_WINDOW,
    event_windows=IMAGE_SPEC_ROBUSTNESS_WINDOWS,
)

car_df = analysis_df[
    [
        "transcriptid",
        "permno",
        "call_date",
        "sentiment_bucket",
        "clarity_bucket",
        "revenue_bucket",
        "buyback_sentiment_mean",
        "clarity_composite",
        "revenue_surprise",
    ]
].copy().merge(event_results, on=["transcriptid", "permno", "call_date"], how="left")
car_df["car"] = car_df["car_1_3"]

matrix_results = create_sentiment_clarity_matrix(car_df, "sentiment_bucket", "clarity_bucket", "car")
three_way_results = create_three_way_sort(
    car_df,
    "sentiment_bucket",
    "clarity_bucket",
    "revenue_bucket",
    "car",
)
caar_by_bins = compute_caar_by_bins(car_df, ["sentiment_bucket", "clarity_bucket", "revenue_bucket"])

format_results_table(matrix_results).head()

In [ ]:
regression_df = car_df.dropna(
    subset=["car", "buyback_sentiment_mean", "clarity_composite", "revenue_surprise"]
).copy()

if not regression_df.empty:
    regression = smf.ols(
        "car ~ buyback_sentiment_mean * clarity_composite + revenue_surprise",
        data=regression_df,
    ).fit()
    print(regression.summary())
else:
    print("Regression skipped because the merged CAR sample is empty.")

paths.outputs_dir.mkdir(exist_ok=True)
tables_dir = paths.outputs_dir / "tables"
tables_dir.mkdir(exist_ok=True)

format_results_table(matrix_results).to_csv(tables_dir / "buyback_sentiment_clarity_matrix.csv", index=False)
format_results_table(three_way_results).to_csv(tables_dir / "buyback_three_way_sort.csv", index=False)
format_results_table(caar_by_bins).to_csv(tables_dir / "buyback_caar_by_bins.csv", index=False)

matrix_results.head()